# The pinhole camera

*graphics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/graphics/camera_projection)

How a 3D scene becomes a 2D image: model, view and projection $4\times4$ matrices act on homogeneous coordinates, then the perspective divide and the viewport give pixels. All vertices of a wireframe object are projected at once with one einsum while the camera orbits. Compare perspective with orthographic projection and with a dolly zoom, see parallel lines meet at vanishing points, and write the same camera as $K[R\,|\,\mathbf t]$.

**Vector form**

$$\tilde{\mathbf x} = P\,V\,M\,\mathbf X,\quad \mathbf X = (x, y, z, 1)^{\mathsf T},\qquad \mathbf x_\text{ndc} = \tilde{\mathbf x}/\tilde x_4,\qquad \lambda\,(u, v, 1)^{\mathsf T} = K\,[R\,|\,\mathbf t]\,\mathbf X,\quad \mathbf v_\infty = K R\,\hat{\mathbf d}$$

**Index (tensor) form**

$$\tilde x_{vi} = P_{ij}V_{jk}M_{kl}X_{vl},\quad u_v = \tfrac W2\big(\tilde x_{v1}/\tilde x_{v4} + 1\big),\quad v_v = \tfrac H2\big(1 - \tilde x_{v2}/\tilde x_{v4}\big)\qquad \lambda_v\,\hat u_{va} = K_{ab}\,[R|t]_{bl}\,X_{vl}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $X_{vl}$ | `tensors.shape(name, device)` | (N, 4) | vertex $v$ in homogeneous coordinates $(x, y, z, 1)$; edges are index pairs $(E, 2)$ |
| $M_{kl}$ | `model_matrix(yaw, scale, lift)` | (4, 4) | object → world: scale, rotate about $y$, then translate (the 4th column) |
| $V_{jk}$ | `look_at(eye, target)` | (4, 4) | world → camera: rows are the camera axes, translation $-R\,\mathbf e$ |
| $P_{ij}$ | `perspective(fov, aspect, near, far)` | (4, 4) | camera → clip space; its last row copies $-z$ into $\tilde x_4$ (orthographic: $\tilde x_4 = 1$) |
| $\tilde x_{vi} = P_{ij}V_{jk}M_{kl}X_{vl}$ | `einsum('ij,jk,kl,vl->vi', P, V, M, X)` | (N, 4) | repeated $j, k, l$ ⇒ summed: every vertex through all three matrices in one call |
| $(u_v, v_v)$ | `to_pixels(clip, W, H)` | (N, 2) | perspective divide by $\tilde x_4$, then the viewport; image rows count downwards |
| $K_{ab},\ [R\vert t]_{bl}$ | `intrinsics(fov, W, H), extrinsics(V)` | (3, 3), (3, 4) | focal length $f = \tfrac H2 / \tan\tfrac{\text{fov}}2$ and principal point; the rigid motion world → camera |
| $K_{ab}R_{bc}d_c$ | `vanishing_point(K, Rt, d)` | (2,) | a point at infinity $(\hat{\mathbf d}, 0)$ loses the translation: all lines along $\hat{\mathbf d}$ meet there |

The 4th coordinate is what makes the camera linear: translation becomes a matrix column, and perspective, which divides by depth, becomes a matrix that copies depth into $\tilde x_4$ followed by one division at the end. Points with $w = 0$ are directions; their images are the vanishing points, and all horizontal directions vanish on the horizon, which sits above the image centre when the camera looks down. The orthographic camera has $\tilde x_4 = 1$, so it has no vanishing points and far objects do not shrink, as in engineering drawings and many strategy games. The dolly zoom (the "Vertigo effect") changes the field of view while moving the camera so that $d\,\tan(\text{fov}/2)$ stays fixed: the object keeps its size but the floor behind it stretches. Vision libraries use $\lambda\,\hat{\mathbf u} = K[R|\mathbf t]\mathbf X$ with $z$ pointing forwards; graphics APIs use $P V M$ with $z$ pointing backwards; the two differ only by flipping the $y$ and $z$ axes, and the terminal checks that they give the same pixels.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/graphics/camera_projection/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.plotting import frame_indices
from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import extrinsics, intrinsics, look_at, model_matrix, perspective, pinhole, to_pixels


def matrix(a: torch.Tensor) -> str:
    """Print a small matrix one row per line."""
    return "\n".join("  [" + "  ".join(f"{x:8.3f}" for x in row) + " ]" for row in a.tolist())


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    w, h, fov = params["width"], params["height"], params["fov"]
    obj, edges = tensors.shape(params["shape"], dev)  # (N, 4), (E, 2)
    grid, grid_edges = tensors.floor_grid(3.0, 7, dev)
    model = model_matrix(params["yaw"], 1.8, 0.9, dev)  # (4, 4): scale 1.8, standing on the floor
    target = torch.tensor([0.0, 0.9, 0.0], device=dev)
    eyes = tensors.orbit(params["frames"], params["distance"], params["height_cam"], dev)
    eyes = eyes[frame_indices(len(eyes), len(eyes))]
    print(f"{params['shape']}: {len(obj)} vertices, {len(edges)} edges; floor grid {len(grid)} vertices")

    view = look_at(eyes[0], target)
    k, rt = intrinsics(fov, w, h, dev), extrinsics(view)
    print(f"intrinsics K (f = {k[0, 0]:.1f} px, principal point ({k[0, 2]:.0f}, {k[1, 2]:.0f})):\n{matrix(k)}")
    print(f"extrinsics [R|t] of the first camera:\n{matrix(rt)}")
    centre = torch.cat([target, torch.ones(1, device=dev)])[None]
    u, v = pinhole(k, rt, centre)[0].tolist()
    print(f"the look-at point lands on pixel ({u:.2f}, {v:.2f}) = the principal point")
    clip = torch.einsum("ij,jk,kl,vl->vi", perspective(fov, w / h, solve.NEAR, solve.FAR, dev), view, model, obj)
    gap = (to_pixels(clip, w, h) - pinhole(k, rt, torch.einsum("kl,vl->vk", model, obj))).abs().max()
    print(f"4×4 pipeline P·V·M vs K[R|t]·M: largest difference {gap:.1e} px")

    res = solve.render(obj, grid, model, eyes, target, fov, w, h)
    vp = res["vp"][0]
    x_vp, z_vp = (f"({a:.0f}, {b:.0f})" for a, b in vp.tolist())
    print(f"vanishing points of the floor lines (frame 0): x-lines at {x_vp}, z-lines at {z_vp} px")
    print(f"horizon at image row v = {res['horizon'][0]:.1f} (above the centre {h / 2:.0f}: the camera looks down)")
    print(
        f"dolly zoom: fov {res['dolly_fov'].min():.0f}°–{res['dolly_fov'].max():.0f}°, "
        f"distance {res['dolly_dist'].min():.2f}–{res['dolly_dist'].max():.2f}"
    )
    return plot.animate(res, edges, grid_edges, w, h, fov, out_dir / "camera_projection.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/graphics/camera_projection/equation.py
r"""Pinhole camera → tensor translation.

A 3D point is a homogeneous 4-vector X = (x, y, z, 1); a direction (a point at infinity) is (d_x, d_y, d_z, 0).
Three 4×4 matrices carry every vertex from the object to the screen (OpenGL convention, camera looking along −z):

    model M (object → world),   view V (world → camera),   projection P (camera → clip space)

    x̃_{vi} = P_{ij} V_{jk} M_{kl} X_{vl}          (j, k, l summed; v runs over all vertices at once)

Perspective divide and viewport turn clip coordinates into pixels (image rows run downwards):

    x^ndc_{va} = x̃_{va} / x̃_{v4},    u_v = W (x^ndc_{v1} + 1)/2,    v_v = H (1 − x^ndc_{v2})/2

Computer vision writes the same camera as λ (u, v, 1) = K [R|t] X: K holds the focal length f (pixels) and the
principal point (c_x, c_y), [R|t] is the rigid motion world → camera. A direction (w = 0) drops t, so
parallel lines with direction d all meet at the vanishing point K R d.
"""

import math

import torch


def normalize(v: torch.Tensor) -> torch.Tensor:
    return v / torch.linalg.vector_norm(v, dim=-1, keepdim=True)


def model_matrix(yaw_deg: float, scale: float, lift: float, device: torch.device) -> torch.Tensor:
    """M = T(0, lift, 0) · R_y(yaw) · S(scale): place the object on the floor.   → (4, 4)"""
    a = math.radians(yaw_deg)
    m = torch.eye(4, device=device)
    m[:3, :3] = scale * torch.tensor(
        [[math.cos(a), 0, math.sin(a)], [0, 1, 0], [-math.sin(a), 0, math.cos(a)]], device=device
    )
    m[1, 3] = lift
    return m


def look_at(eye: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """View matrix V: rows right, up, −forward of the camera, then translate the eye to the origin.   → (4, 4)"""
    f = normalize(target - eye)
    r = normalize(torch.linalg.cross(f, torch.tensor([0.0, 1.0, 0.0], device=eye.device)))
    u = torch.linalg.cross(r, f)
    v = torch.eye(4, device=eye.device)
    v[:3, :3] = torch.stack([r, u, -f])
    v[:3, 3] = -v[:3, :3] @ eye  # t = −R · eye
    return v


def perspective(fov_deg: float, aspect: float, near: float, far: float, device: torch.device) -> torch.Tensor:
    """Perspective projection P: x̃_4 = −z_cam, so dividing by it shrinks far objects.   → (4, 4)"""
    f = 1 / math.tan(math.radians(fov_deg) / 2)
    return torch.tensor(
        [
            [f / aspect, 0, 0, 0],
            [0, f, 0, 0],
            [0, 0, (far + near) / (near - far), 2 * far * near / (near - far)],
            [0, 0, -1, 0],
        ],
        device=device,
    )


def orthographic(half_height: float, aspect: float, near: float, far: float, device: torch.device) -> torch.Tensor:
    """Orthographic projection: x̃_4 = 1 always, so size does not depend on distance.   → (4, 4)"""
    return torch.tensor(
        [
            [1 / (aspect * half_height), 0, 0, 0],
            [0, 1 / half_height, 0, 0],
            [0, 0, -2 / (far - near), -(far + near) / (far - near)],
            [0, 0, 0, 1],
        ],
        device=device,
    )


def clip_coords(p: torch.Tensor, v: torch.Tensor, m: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    r"""x̃_{vi} = P_{ij} V_{jk} M_{kl} X_{vl}   (j, k, l summed).   X: (N, 4) → (N, 4)"""
    return torch.einsum("ij,jk,kl,vl->vi", p, v, m, x)


def to_pixels(clip: torch.Tensor, width: int, height: int) -> torch.Tensor:
    """Perspective divide x̃_a / x̃_4, then the viewport map to pixels (u right, v down).   (N, 4) → (N, 2)"""
    ndc = clip[:, :2] / clip[:, 3:4]
    return torch.stack([width * (ndc[:, 0] + 1) / 2, height * (1 - ndc[:, 1]) / 2], dim=-1)


def intrinsics(fov_deg: float, width: int, height: int, device: torch.device) -> torch.Tensor:
    """K: focal length f = (H/2)/tan(fov/2) in pixels and principal point (W/2, H/2).   → (3, 3)"""
    f = height / 2 / math.tan(math.radians(fov_deg) / 2)
    return torch.tensor([[f, 0, width / 2], [0, f, height / 2], [0, 0, 1]], device=device)


def extrinsics(view: torch.Tensor) -> torch.Tensor:
    """[R|t] in the vision convention (x right, y down, z forward): flip the y and z rows of V.   → (3, 4)"""
    flip = torch.diag(torch.tensor([1.0, -1.0, -1.0], device=view.device))
    return flip @ view[:3]


def pinhole(k: torch.Tensor, rt: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    r"""λ_v û_{va} = K_{ab} [R|t]_{bl} X_{vl}   (b, l summed), then divide by λ_v = depth.   (N, 4) → (N, 2)"""
    h = torch.einsum("ab,bl,vl->va", k, rt, x)  # (N, 3)
    return h[:, :2] / h[:, 2:3]


def vanishing_point(k: torch.Tensor, rt: torch.Tensor, direction: torch.Tensor) -> torch.Tensor:
    """Image of the point at infinity (d, 0): where all lines with direction d meet.   (3,) → (2,)"""
    x = torch.cat([direction, direction.new_zeros(1)])[None]  # (1, 4) with w = 0
    return pinhole(k, rt, x)[0]


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/graphics/camera_projection/tensors.py
"""Build the concrete tensors: wireframe objects and the floor grid as homogeneous vertices + edge lists."""

import math

import torch


def homogeneous(xyz: torch.Tensor) -> torch.Tensor:
    """Append w = 1 to every point: (N, 3) → (N, 4)."""
    return torch.cat([xyz, torch.ones(len(xyz), 1, device=xyz.device)], dim=1)


def cube(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Unit cube centred at the origin: vertices (8, 4), edges (12, 2) as vertex index pairs."""
    corners = torch.tensor([[x, y, z] for x in (-0.5, 0.5) for y in (-0.5, 0.5) for z in (-0.5, 0.5)], device=device)
    edges = [(a, b) for a in range(8) for b in range(a + 1, 8) if bin(a ^ b).count("1") == 1]  # differ in one axis
    return homogeneous(corners), torch.tensor(edges, device=device)


def house(device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """The cube with a gable roof: two ridge points above the top face.   vertices (10, 4), edges (17, 2)."""
    v, e = cube(device)
    ridge = homogeneous(torch.tensor([[-0.5, 1.0, 0.0], [0.5, 1.0, 0.0]], device=device))
    # top corners: 2 (−x,+y,−z), 3 (−x,+y,+z), 6 (+x,+y,−z), 7 (+x,+y,+z); ridge points are 8 (−x) and 9 (+x)
    roof = torch.tensor([[8, 9], [8, 2], [8, 3], [9, 6], [9, 7]], device=device)
    return torch.cat([v, ridge]), torch.cat([e, roof])


def torus(device: torch.device, big: float = 0.5, small: float = 0.2, n_u: int = 20, n_v: int = 8):
    """Torus around the y axis: vertices (n_u·n_v, 4) and edges along both rings (2·n_u·n_v, 2)."""
    u = torch.arange(n_u, device=device) * 2 * math.pi / n_u
    v = torch.arange(n_v, device=device) * 2 * math.pi / n_v
    uu, vv = torch.meshgrid(u, v, indexing="ij")  # (n_u, n_v)
    rho = big + small * torch.cos(vv)
    xyz = torch.stack([rho * torch.cos(uu), small * torch.sin(vv), rho * torch.sin(uu)], dim=-1).reshape(-1, 3)
    idx = torch.arange(n_u * n_v, device=device).reshape(n_u, n_v)
    around = torch.stack([idx, idx.roll(-1, dims=1)], dim=-1).reshape(-1, 2)  # small circles
    along = torch.stack([idx, idx.roll(-1, dims=0)], dim=-1).reshape(-1, 2)  # big circles
    return homogeneous(xyz), torch.cat([around, along])


def shape(name: str, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    return {"cube": cube, "house": house, "torus": torus}[name](device)


def floor_grid(extent: float, n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Lines x = const and z = const on the floor y = 0: vertices (4n, 4), edges (2n, 2).
    The first n edges run along z, the last n along x, so each family is a set of parallel lines."""
    s = torch.linspace(-extent, extent, n, device=device)
    e = torch.full_like(s, extent)
    zero = torch.zeros_like(s)
    along_z = torch.cat([torch.stack([s, zero, -e], 1), torch.stack([s, zero, e], 1)])  # (2n, 3)
    along_x = torch.cat([torch.stack([-e, zero, s], 1), torch.stack([e, zero, s], 1)])
    k = torch.arange(n, device=device)
    edges = torch.cat([torch.stack([k, k + n], 1), torch.stack([k + 2 * n, k + 3 * n], 1)])
    return homogeneous(torch.cat([along_z, along_x])), edges


def orbit(n_frames: int, distance: float, height: float, device: torch.device) -> torch.Tensor:
    """Camera positions on a circle around the y axis, starting 30° off the z axis: (F, 3)."""
    a = math.radians(30) + torch.linspace(0, 2 * math.pi, n_frames + 1, device=device)[:-1]
    return torch.stack([distance * torch.sin(a), torch.full_like(a, height), distance * torch.cos(a)], dim=-1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/graphics/camera_projection/solve.py
"""Project the object and the floor grid for every camera position, in three cameras side by side."""

import math

import torch

from problems._common.progress import steps

from .equation import (
    clip_coords,
    extrinsics,
    intrinsics,
    look_at,
    orthographic,
    perspective,
    to_pixels,
    vanishing_point,
)

NEAR, FAR = 0.1, 100.0


def project(p: torch.Tensor, v: torch.Tensor, m: torch.Tensor, x: torch.Tensor, w: int, h: int):
    """Pixels (N, 2) of homogeneous points x (N, 4), and a mask (N,) of points in front of the camera."""
    clip = clip_coords(p, v, m, x)
    return to_pixels(clip, w, h), clip[:, 3] > NEAR


def dolly(fov0: float, distance: float, n_frames: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Dolly zoom: the field of view swings between 20° and 80° while the camera moves so that
    distance · tan(fov/2) stays fixed, keeping the object the same size.   → fov (F,), distance (F,)"""
    phase = torch.linspace(0, 2 * math.pi, n_frames + 1)[:-1]
    fov = 50 - 30 * torch.cos(phase)
    dist = distance * math.tan(math.radians(fov0) / 2) / torch.tan(torch.deg2rad(fov) / 2)
    return fov, dist


def render(
    obj: torch.Tensor,
    grid: torch.Tensor,
    model: torch.Tensor,
    eyes: torch.Tensor,
    target: torch.Tensor,
    fov: float,
    w: int,
    h: int,
) -> dict[str, torch.Tensor]:
    """For every frame: object and grid pixels in the perspective, orthographic and dolly-zoom cameras,
    and the vanishing points of the two grid directions (x and z) plus the horizon row of the perspective view."""
    dev = obj.device
    eye_grid = torch.eye(4, device=dev)  # the grid is already in world coordinates: M = I
    aspect = w / h
    persp = perspective(fov, aspect, NEAR, FAR, dev)
    dolly_fov, dolly_dist = dolly(fov, torch.linalg.vector_norm(eyes[0] - target).item(), len(eyes))
    k = intrinsics(fov, w, h, dev)
    out = {name: [] for name in ("obj", "obj_ok", "grid", "grid_ok", "vp", "horizon")}
    for f in steps(len(eyes), "frames"):
        view = look_at(eyes[f], target)
        dist = torch.linalg.vector_norm(eyes[f] - target).item()
        ortho = orthographic(dist * math.tan(math.radians(fov) / 2), aspect, NEAR, FAR, dev)  # same size at target
        d_eye = target + (eyes[f] - target) * dolly_dist[f].item() / dist
        cams = [
            (persp, view),
            (ortho, view),
            (perspective(dolly_fov[f].item(), aspect, NEAR, FAR, dev), look_at(d_eye, target)),
        ]
        for name, x, m in (("obj", obj, model), ("grid", grid, eye_grid)):
            px, ok = zip(*(project(p, v, m, x, w, h) for p, v in cams))
            out[name].append(torch.stack(px))  # (3, N, 2)
            out[name + "_ok"].append(torch.stack(ok))  # (3, N)

        rt = extrinsics(view)
        axes = torch.eye(3, device=dev)
        out["vp"].append(torch.stack([vanishing_point(k, rt, axes[0]), vanishing_point(k, rt, axes[2])]))  # (2, 2)
        ahead = target - eyes[f]
        ahead[1] = 0  # the horizontal direction straight ahead: its vanishing point lies on the horizon
        out["horizon"].append(vanishing_point(k, rt, ahead)[1])
    result = {name: torch.stack(v) for name, v in out.items()}
    result["dolly_fov"], result["dolly_dist"] = dolly_fov, dolly_dist
    return result


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/graphics/camera_projection/plot.py
"""Three cameras side by side: perspective (with vanishing points), orthographic, and a dolly zoom."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle

from problems._common.plotting import save_gif


def segments(px: np.ndarray, ok: np.ndarray, edges: np.ndarray) -> np.ndarray:
    """Line segments (E', 2, 2) for the edges whose two ends are in front of the camera."""
    keep = ok[edges[:, 0]] & ok[edges[:, 1]]
    return px[edges[keep]]


def animate(res: dict, edges: torch.Tensor, grid_edges: torch.Tensor, w: int, h: int, fov: float, path: Path) -> Path:
    r = {k: v.cpu().numpy() for k, v in res.items()}
    e, ge = edges.cpu().numpy(), grid_edges.cpu().numpy()
    n_lines = len(ge) // 2  # first half of the grid edges run along z, second half along x

    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4), gridspec_kw={"width_ratios": [1.6, 1, 1]})
    titles = ["perspective", "orthographic", "dolly zoom"]
    grids, objs = [], []
    for c, (ax, title) in enumerate(zip(axes, titles)):
        frame = ax.add_patch(Rectangle((0, 0), w, h, facecolor="#f9fafb", edgecolor="#374151", lw=1.2, zorder=0))
        grids.append(ax.add_collection(LineCollection([], colors="#9ca3af", lw=0.8, zorder=2)))
        objs.append(ax.add_collection(LineCollection([], colors="#2563eb", lw=1.6, zorder=3)))
        if c == 0:  # room around the image for the vanishing points
            ax.set_xlim(-0.6 * w, 1.6 * w)
            ax.set_ylim(1.2 * h, -0.45 * h)  # image rows grow downwards
        else:  # just the image: clip the lines at its border
            grids[c].set_clip_path(frame)
            objs[c].set_clip_path(frame)
            ax.set_xlim(-0.03 * w, 1.03 * w)
            ax.set_ylim(1.03 * h, -0.03 * h)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        ax.set_title(title, fontsize=10)
    guides = axes[0].add_collection(LineCollection([], colors="#dc2626", lw=0.6, linestyles="--", alpha=0.5, zorder=1))
    (vps,) = axes[0].plot([], [], "o", color="#dc2626", ms=6, label="vanishing points")
    (horizon,) = axes[0].plot([], [], "--", color="#f59e0b", lw=1.2, label="horizon")
    axes[0].legend(loc="lower left", fontsize=7, framealpha=0.9)
    axes[1].text(0.5, 0.04, "parallel lines stay parallel", transform=axes[1].transAxes, ha="center", fontsize=8)
    dolly_label = axes[2].text(0.5, 0.04, "", transform=axes[2].transAxes, ha="center", fontsize=8)
    axes[0].set_title(f"perspective, fov {fov:.0f}°", fontsize=10)

    def update(f: int):
        for c in range(3):
            grids[c].set_segments(segments(r["grid"][f, c], r["grid_ok"][f, c], ge))
            objs[c].set_segments(segments(r["obj"][f, c], r["obj_ok"][f, c], e))
        # extend every floor line from its end nearer the vanishing point of its family (z-lines → vp[1], x → vp[0])
        px, vp = r["grid"][f, 0], r["vp"][f]
        lines = []
        for j, (a, b) in enumerate(ge):
            target = vp[1] if j < n_lines else vp[0]
            if np.all(np.isfinite(target)):
                near = px[a] if np.hypot(*(px[a] - target)) < np.hypot(*(px[b] - target)) else px[b]
                lines.append([near, target])
        guides.set_segments(lines)
        vps.set_data(vp[:, 0], vp[:, 1])
        horizon.set_data([-w, 2 * w], [r["horizon"][f]] * 2)
        dolly_label.set_text(f"fov {r['dolly_fov'][f]:.0f}°, distance {r['dolly_dist'][f]:.1f}")

    fig.tight_layout()
    return save_gif(fig, update, list(range(len(r["vp"]))), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'shape': 'house',
    'yaw': 0.0,
    'fov': 50.0,
    'distance': 6.5,
    'height_cam': 2.5,
    'width': 320,
    'height': 240,
    'frames': 72
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.graphics.camera_projection")]:
    del sys.modules[name]
main = importlib.import_module("problems.graphics.camera_projection.main")
problem_dir = Path("problems/graphics/camera_projection")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
